# KAT-VAD v2 — **MM-AU / CAP-DATA Phase 0**: is DoTA inside CAP? (no training)

Plan `.project/plans/katvad-mmau-phase0.md` — the matching rule (§3) and the D9 branch rule (§4) were fixed **before
any number**. CAP-DATA (`JeffreyChou/MM-AU` on HF, 9,768 accident videos, 267 GB) ships **no provenance column**, so
provenance is read from content: every DoTA clip (`DoTA_s1_ncc`) and every DADA-2000 source (`DADA2000_orig`) is
looked up in a CAP CLIP cache built with the **same** transform (`no_center_crop`) at stride 1.

| step | what | where |
|---|---|---|
| 0 | setup, deps, annotation census | CPU |
| 1 | **P0a** mirror the CAP parts of `GROUPS` HF → Drive (`Thesis/data/MMAU/raw/`), size-check every part | CPU, network |
| 2 | **P0b/P0c** per group: stream `cat part_* \| tar -xz` to VM disk → census → CLIP s1 `_ncc` → Drive → delete frames | **GPU (L4)** |
| 3 | `core.tools.mmau_match` → read-out + branch | CPU |

**Run it twice.** First with `GROUPS = ['1-10']` (P0b: feasibility only, the tool prints `FEASIBILITY_ONLY`), paste
the read-out; then with all five groups (P0c: the branch). Every step is resumable: parts already on Drive are
skipped, groups whose CLIP files all exist are skipped, `extract_clip_features` resumes per video.

**Not downloaded:** `DADA-2000_chunks` (= the T2 corpus, already on Drive) and the detection subsets.
DoTA-eval stays sealed: only DoTA-dev is read through `load_split`. No label and no score is read.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'MMAU_MATCH_EXACT'), 'this upload predates the MM-AU P0 code -- re-upload branch v2'

# ---- which CAP groups this run covers: ['1-10'] first (P0b), then all five (P0c) ----
GROUPS = ['1-10']
# GROUPS = ['1-10', '11', '12-42', '43', '44-62']

# README of the HF repo: sequences and extracted size (GB) per group
EXPECTED = {'1-10': (1556, 44), '11': (3083, 96), '12-42': (1629, 45), '43': (2150, 44), '44-62': (1350, 30)}
assert set(GROUPS) <= set(EXPECTED), GROUPS

RAW       = constants.DATA_ROOT / 'MMAU' / 'raw'
CAP_CLIP  = constants.CLIP_CACHE_DIR / f'{constants.MMAU_CAP_DATASET}_s1_ncc'
DOTA_S1   = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
DADA_CLIP = constants.CLIP_CACHE_DIR / constants.DADA_ORIGIN_DATASET
REPORT    = constants.OUTPUT_ROOT / 'REPORTS' / 'mmau_p0'
LOCAL     = Path('/content/mmau')
for p in (RAW, CAP_CLIP, REPORT, LOCAL):
    p.mkdir(parents=True, exist_ok=True)
for name, path in (('DoTA s1', DOTA_S1), ('DADA clip', DADA_CLIP)):
    assert path.is_dir(), f'missing {name}: {path}'
    print(f'  {name:10s} {path}   OK')
print('groups this run:', GROUPS)

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu "huggingface_hub>=0.25" hf_transfer xlrd
python -c "import av, einops, faiss, xlrd, hf_transfer; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 2 needs one'
df -h /content | tail -1

## 1. P0a — mirror CAP parts HF → Drive, size-check every part

Server-to-server (HF → Colab → Drive). Kept on Drive on purpose: when a link dies the pixels are gone (DoTA).

In [ ]:
import json
import time

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import HfApi, snapshot_download  # noqa: E402

api = HfApi()
remote = {}
for g in GROUPS:
    for f in api.list_repo_tree(constants.MMAU_HF_REPO, repo_type='dataset',
                                path_in_repo=f'CAP-DATA_chunks/{g}', expand=True):
        if getattr(f, 'size', None) is not None:
            remote[f.path] = f.size
print(f'{len(remote)} parts, {sum(remote.values()) / 1e9:.1f} GB for {GROUPS}')

t0 = time.time()
snapshot_download(constants.MMAU_HF_REPO, repo_type='dataset', local_dir=str(RAW), max_workers=8,
                  allow_patterns=[f'CAP-DATA_chunks/{g}/*' for g in GROUPS]
                  + ['cap_text_annotations.xls', 'README.md'])
print(f'download step {time.time() - t0:.0f}s')

bad = {p: (s, (RAW / p).stat().st_size if (RAW / p).exists() else None)
       for p, s in remote.items() if not (RAW / p).exists() or (RAW / p).stat().st_size != s}
assert not bad, f'{len(bad)} parts missing or wrong size: {list(bad.items())[:5]}'
manifest_path = RAW / 'cap_parts_manifest.json'
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
manifest.update(remote)
manifest_path.write_text(json.dumps(manifest, indent=1, sort_keys=True))
print(f'all {len(remote)} parts match HF sizes; manifest {manifest_path} ({len(manifest)} parts total)')

In [ ]:
import pandas as pd

ann = pd.read_excel(RAW / 'cap_text_annotations.xls', sheet_name='annotation file')
ann_frames = dict(zip(ann['video'].astype(int), ann['total frames'].astype(int)))
assert len(ann_frames) == constants.MMAU_CAP_VIDEOS, (len(ann_frames), constants.MMAU_CAP_VIDEOS)
print(f'annotation: {len(ann_frames)} videos, {sum(ann_frames.values())} frames, '
      f'accident flag {ann["whether an accident occurred (1/0)"].value_counts().to_dict()}')

## 2. Per group: extract to VM disk → census → CLIP s1 `_ncc` → delete frames

The tar is streamed (`cat part_* | tar -xz`), never merged into one file. Frames live on `/content` only
(millions of small JPEGs on Drive FUSE are slow and leave empty folders, lesson C10).

In [ ]:
import shutil
import subprocess

from core.data.video_io import list_frame_folders, list_frame_images  # noqa: E402


def stream(cmd, name):
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print(''.join(tail))
        raise RuntimeError(f'{name} exited {proc.returncode}')


census = {}
for g in GROUPS:
    n_expected, gb = EXPECTED[g]
    cached = {p.stem for p in CAP_CLIP.glob('*.npy')}
    lo, hi = (int(x) for x in (g.split('-') * 2)[:2])
    ann_ids = {f'{v:06d}' for v, t in zip(ann['video'].astype(int), ann['type'].astype(int)) if lo <= t <= hi}
    if ann_ids and ann_ids <= cached:
        print(f'group {g}: all {len(ann_ids)} annotated videos already cached -- skipped')
        continue
    dest = LOCAL / g
    if dest.exists():
        folders = list_frame_folders(dest, 'images')
    else:
        free_gb = shutil.disk_usage('/content').free / 1e9
        need_gb = 1.1 * gb
        assert free_gb >= need_gb, (f'group {g}: {free_gb:.0f} GB free on /content, need {need_gb:.0f} -- '
                                    'switch to a larger runtime (A100) before extracting')
        parts = sorted((RAW / 'CAP-DATA_chunks' / g).glob(f'{g}.part_*'))
        assert parts, f'no parts for group {g}'
        dest.mkdir(parents=True)
        t0 = time.time()
        subprocess.run(['bash', '-o', 'pipefail', '-c',
                        'cat "$@" | tar -xzf - -C ' + str(dest), '_', *map(str, parts)], check=True)
        print(f'group {g}: extracted {len(parts)} parts in {time.time() - t0:.0f}s')
        folders = list_frame_folders(dest, 'images')

    ids = [f.name for f in folders]
    frames = {f.name: len(list_frame_images(f, 'images')) for f in folders}
    off = {v: (n, ann_frames.get(int(v))) for v, n in frames.items() if ann_frames.get(int(v)) != n}
    census[g] = {'folders': len(ids), 'expected': n_expected, 'frames': sum(frames.values()),
                 'not_in_annotation': sum(int(v) not in ann_frames for v in ids),
                 'frame_count_mismatch': len(off), 'empty': sum(n == 0 for n in frames.values())}
    print(f'census {g}: {census[g]}; first mismatches {list(off.items())[:5]}')
    assert len(set(ids)) == len(ids), f'duplicate video ids inside group {g}'

    todo = [v for v in ids if v not in cached and frames[v] > 0]
    print(f'group {g}: {len(ids) - len(todo)} / {len(ids)} already cached, {len(todo)} to encode')
    if todo:
        stream([sys.executable, '-m', 'core.tools.extract_clip_features',
                '--frames-dir', dest, '--frames-subdir', 'images',
                '--dataset', constants.MMAU_CAP_DATASET, '--output-dir', CAP_CLIP,
                '--stride', '1', '--no-center-crop', '--batch-size', '256', '--device', 'auto'],
               'extract_clip_features')
    done = {p.stem for p in CAP_CLIP.glob('*.npy')}
    missing = [v for v in ids if frames[v] > 0 and v not in done]
    assert not missing, f'group {g}: {len(missing)} videos not cached, e.g. {missing[:5]}'
    shutil.rmtree(dest)
    print(f'group {g}: done, frames deleted; free {shutil.disk_usage("/content").free / 1e9:.0f} GB')

(REPORT / 'census.json').write_text(json.dumps(census, indent=1))

## 3. Match → read-out → branch (plan §3–§4)

Caches are staged to VM disk first (reading ~10k small `.npy` from Drive one by one inside the tool is slow).
With fewer than 9,768 CAP clips cached the tool reads **`FEASIBILITY_ONLY`** — no branch is decided on one group.

In [ ]:
t0 = time.time()
stage = LOCAL / 'stage'
for src, name in ((CAP_CLIP, 'cap'), (DOTA_S1, 'dota_s1'), (DADA_CLIP, 'dada')):
    dst = stage / name
    dst.mkdir(parents=True, exist_ok=True)
    have = {p.name for p in dst.glob('*.npy')}
    for p in src.glob('*.npy'):
        if p.name not in have:
            shutil.copy2(p, dst / p.name)
print(f'staged in {time.time() - t0:.0f}s: ' + ', '.join(
    f'{n} {len(list((stage / n).glob("*.npy")))}' for n in ('cap', 'dota_s1', 'dada')))

n_cap = len(list((stage / 'cap').glob('*.npy')))
OUT = REPORT / ('all' if n_cap >= constants.MMAU_CAP_VIDEOS else 'group_' + '+'.join(GROUPS))
stream([sys.executable, '-m', 'core.tools.mmau_match',
        '--cap-clip-dir', stage / 'cap', '--dota-clip-dir', stage / 'dota_s1',
        '--dada-clip-dir', stage / 'dada', '--out-dir', OUT], 'mmau_match')
print((OUT / 'mmau_p0_readout.md').read_text())
print('census:', json.dumps(census))

## 4. Record

Paste the read-out above (and the census line) back into the chat. Nothing here is a decision until the full-CAP run
(`OUT = .../mmau_p0/all`) prints a branch; then the branch goes into Amendment 4 (D9) per plan §4.